In [5]:
# Carga del modelo
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
# Vectorización
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [4]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [5]:
# Producto vectorial para calcular la similaridad mediante coseno. Funciona ya que el modelo está normalizando
# los vectores a longitud de la unidad. Más info: https://en.wikipedia.org/wiki/Cosine_similarity
v1.dot(dv)


np.float32(0.32332397)

In [6]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [7]:
v2.dot(dv)


np.float32(0.01973048)

In [1]:
# Cargamos los datos del faq para poder probar su vectorización
from ingest import load_faq_data
documents = load_faq_data();

In [2]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text);

In [3]:
# Se usa apra la barra de progreso.
from tqdm.auto import tqdm


In [6]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [8]:
# Generamos una matriz con numpy , y mostramos su forma.
import numpy as np
X = np.array(vectors)

In [9]:
X.shape

(1350, 384)

In [19]:
# Consulta vectorial a todos los elementos del FAQ vectorizados
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

# Es óptimo llamarlo así porque hace el for con C y no con python.
scores = X.dot(v_query)


In [18]:
# Vector con mayor similaridad
idx = np.argmax(scores)
idx, scores[idx], documents[idx]

(np.int64(2),
 np.float32(0.76294094),
 {'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'doc_id': '3f1424af17'})

In [20]:
# TOp 5 similaridades
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [21]:
scores[top5]

array([0.76294094, 0.7579371 , 0.7192132 , 0.6536312 , 0.56009996],
      dtype=float32)

In [23]:
# Forma más rápida de conseguir el top5.
top5 = np.argsort(-scores)[:5]

In [25]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()


# Obtener varios documentos en vez de uno ayuda en el sentido de que la respuesta a una pregunta puede estar
# en varios documentos.

0.76294094
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579371
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192132
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions'